In [0]:
import os
import requests
from datetime import date, datetime, timedelta

dbutils.widgets.text("start_date", "2026-09-01")
dbutils.widgets.text("end_date", "2026-09-07")

START_DATE = datetime.strptime(
    dbutils.widgets.get("start_date"),
    "%Y-%m-%d"
).date()

END_DATE = datetime.strptime(
    dbutils.widgets.get("end_date"),
    "%Y-%m-%d"
).date()

BASE_URL = "https://data.gharchive.org"
LANDING_DIR = "/Volumes/workspace/default/gharchive_raw"

if START_DATE > END_DATE:
    raise ValueError("start_date must be before or equal to end_date")

print("Download range:", START_DATE, "to", END_DATE)

In [0]:
downloaded_count = 0
skipped_count = 0
failed_files = []

current_date = START_DATE

while current_date <= END_DATE:
    date_str = current_date.isoformat()

    for hour in range(24):
        file_name = f"{date_str}-{hour}.json.gz"
        url = f"{BASE_URL}/{file_name}"
        target_path = f"{LANDING_DIR}/{file_name}"

        if os.path.exists(target_path):
            skipped_count += 1
            print(f"SKIP already exists: {file_name}")
            continue

        try:
            print(f"Downloading: {file_name}")

            response = requests.get(
                url,
                stream=True,
                timeout=120
            )

            response.raise_for_status()

            with open(target_path, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

            downloaded_count += 1
            print(f"Saved: {file_name}")

        except Exception as e:
            failed_files.append(
                (file_name, str(e))
            )
            print(f"FAILED: {file_name} -> {e}")

    current_date += timedelta(days=1)

In [0]:
print("Downloaded files:", downloaded_count)
print("Skipped existing files:", skipped_count)
print("Failed files:", len(failed_files))

if failed_files:
    for file_name, error_message in failed_files:
        print(file_name, "->", error_message)

In [0]:
expected_files = 7 * 24

actual_files = 0
missing_files = []

current_date = date(2026, 9, 1)

while current_date <= date(2026, 9, 7):

    date_str = current_date.isoformat()

    for hour in range(24):

        file_name = f"{date_str}-{hour}.json.gz"
        target_path = f"{LANDING_DIR}/{file_name}"

        if os.path.exists(target_path):
            actual_files += 1
        else:
            missing_files.append(file_name)

    current_date += timedelta(days=1)

print("Expected files:", expected_files)
print("Available files:", actual_files)
print("Missing files:", len(missing_files))

if missing_files:
    print(missing_files)